# Models for Future value prediction and to find if its a good investment

### Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import (LinearRegression,Ridge,Lasso,LogisticRegression)
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor,RandomForestClassifier,GradientBoostingClassifier)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor,XGBClassifier
from lightgbm import LGBMRegressor,LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    precision_score,
    f1_score,
    recall_score,
    roc_curve,
    roc_auc_score,
    accuracy_score
)
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import mlflow.xgboost
import mlflow.lightgbm

### Data Load and Train Test Split

In [4]:
df = pd.read_csv(r'Datasets/india_housing_prices_with_target_columns_encoded.csv')
df.head()

,State,Locality,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Availability_Status,Growth_Rate_Annual,Future_Price_5Y,Amenities_Count,Good_Investment,City_Ahmedabad,City_Amritsar,City_Bangalore,City_Bhopal,City_Bhubaneswar,City_Bilaspur,City_Chennai,City_Coimbatore,City_Cuttack,City_Dehradun,City_Durgapur,City_Dwarka,City_Faridabad,City_Gaya,City_Gurgaon,City_Guwahati,City_Haridwar,City_Hyderabad,City_Indore,City_Jaipur,City_Jamshedpur,City_Jodhpur,City_Kochi,City_Kolkata,City_Lucknow,City_Ludhiana,City_Mangalore,City_Mumbai,City_Mysore,City_Nagpur,City_New_Delhi,City_Noida,City_Patna,City_Pune,City_Raipur,City_Ranchi,City_Silchar,City_Surat,City_Trivandrum,City_Vijayawada,City_Vishakhapatnam,City_Warangal,Property_Type_Apartment,Property_Type_Independent_House,Property_Type_Villa,Facing_East,Facing_North,Facing_South,Facing_West,Furnished_Status_Furnished,Furnished_Status_Semi_Furnished,Furnished_Status_Unfurnished,Owner_Type_Broker,Owner_Type_Builder,Owner_Type_Owner
0,Tamil Nadu,Locality_84,1,4740,489.76,0.10,1990,22,1,35,10,3,2,0,0,"Playground, Gym, Garden, Pool, Clubhouse",0,0.042985,604.468164,5,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1
1,Maharashtra,Locality_490,3,2364,195.52,0.08,2008,21,20,17,8,1,0,0,1,"Playground, Clubhouse, Pool, Gym, Garden",1,0.072780,277.808231,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,1,0
2,Punjab,Locality_167,2,3642,183.79,0.05,1997,19,27,28,9,8,0,1,0,"Clubhouse, Pool, Playground, Gym",0,0.041508,225.234639,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,1,0,0
3,Rajasthan,Locality_393,2,2741,300.29,0.11,1991,21,26,34,5,7,2,1,1,"Playground, Clubhouse, Gym, Pool, Garden",0,0.042930,370.524572,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0
4,Rajasthan,Locality_466,4,4823,182.90,0.04,2002,3,2,23,4,9,0,0,1,"Playground, Garden, Gym, Pool, Clubhouse",0,0.030381,212.424051,5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,1,0


In [18]:
x_reg_cols = ['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Floor_No', 'Total_Floors',
    'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals',
    'Public_Transport_Accessibility', 'Parking_Space', 'Security',
    'Availability_Status', 'Amenities_Count',
    'City_Ahmedabad', 'City_Amritsar', 'City_Bangalore', 'City_Bhopal',
    'City_Bhubaneswar', 'City_Bilaspur', 'City_Chennai', 'City_Coimbatore',
    'City_Cuttack', 'City_Dehradun', 'City_Durgapur', 'City_Dwarka',
    'City_Faridabad', 'City_Gaya', 'City_Gurgaon', 'City_Guwahati',
    'City_Haridwar', 'City_Hyderabad', 'City_Indore', 'City_Jaipur',
    'City_Jamshedpur', 'City_Jodhpur', 'City_Kochi', 'City_Kolkata',
    'City_Lucknow', 'City_Ludhiana', 'City_Mangalore', 'City_Mumbai',
    'City_Mysore', 'City_Nagpur', 'City_New_Delhi', 'City_Noida',
    'City_Patna', 'City_Pune', 'City_Raipur', 'City_Ranchi', 'City_Silchar',
    'City_Surat', 'City_Trivandrum', 'City_Vijayawada',
    'City_Vishakhapatnam', 'City_Warangal', 'Property_Type_Apartment',
    'Property_Type_Independent_House', 'Property_Type_Villa', 'Facing_East',
    'Facing_North', 'Facing_South', 'Facing_West',
    'Furnished_Status_Furnished', 'Furnished_Status_Semi_Furnished',
    'Furnished_Status_Unfurnished', 'Owner_Type_Broker',
    'Owner_Type_Builder', 'Owner_Type_Owner']
X_reg = df[x_reg_cols]
y_reg = df['Future_Price_5Y']

In [19]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

In [20]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reg)
X_test_scaled = scaler.transform(X_test_reg)

In [21]:
X_train_reg_scaled = pd.DataFrame(scaler.fit_transform(X_train_reg),
                              columns=X_train_reg.columns, index=X_train_reg.index)
X_test_reg_scaled = pd.DataFrame(scaler.transform(X_test_reg),
                             columns=X_test_reg.columns, index=X_test_reg.index)

In [22]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42),
}

In [23]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("real_estate_regression")

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1787666057911, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1787666057911, lifecycle_stage='active', name='real_estate_regression', tags={}, trace_location=None, workspace='default'>

In [24]:
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_reg_scaled, y_train_reg)
        preds = model.predict(X_test_reg_scaled)

        rmse = np.sqrt(mean_squared_error(y_test_reg, preds))
        mae = mean_absolute_error(y_test_reg, preds)
        r2 = r2_score(y_test_reg, preds)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        signature = infer_signature(X_test_reg_scaled, preds)
        if name == "XGBoost":
            mlflow.xgboost.log_model(model, name="model", signature=signature)
        elif name == "LightGBM":
            mlflow.lightgbm.log_model(model, name="model", signature=signature)
        else:
            mlflow.sklearn.log_model(model, name="model", signature=signature)

        print(f"{name}: RMSE={rmse:.2f}  MAE={mae:.2f}  R2={r2:.3f}")

LinearRegression: RMSE=39.05  MAE=29.32  R2=0.960
🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/4/runs/5354295fe4b2494c957d1cd5728875a5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Ridge: RMSE=39.05  MAE=29.32  R2=0.960
🏃 View run Ridge at: http://127.0.0.1:5000/#/experiments/4/runs/da7189466c6f46e984a9aaa99ebe09c0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Lasso: RMSE=39.57  MAE=29.08  R2=0.959
🏃 View run Lasso at: http://127.0.0.1:5000/#/experiments/4/runs/38fb26e9049b4eaeb319d5d1d4ebbafc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
RandomForest: RMSE=35.43  MAE=24.71  R2=0.967
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/4/runs/c3205c3ca300496a8566c4c180257bbe
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
GradientBoosting: RMSE=37.87  MAE=26.61  R2=0.963
🏃 View run GradientBoosting at: http://127.0.0.1:5000/#/experiments/4/runs/75a83a8730c440ed91736138092834e9
🧪 View experiment at: 

In [25]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [-1, 5, 10, 20],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "num_leaves": [31, 50, 70, 100],
    "subsample": [0.7, 0.8, 1.0],
}

lgbm = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train_reg_scaled, y_train_reg)

print("Best params:", search.best_params_)
print("Best CV R2:", round(search.best_score_, 4))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 200, 'max_depth': -1, 'learning_rate': 0.05}
Best CV R2: 0.9714


In [26]:
best_lgbm = search.best_estimator_

preds = best_lgbm.predict(X_test_reg_scaled)
rmse = np.sqrt(mean_squared_error(y_test_reg, preds))
mae = mean_absolute_error(y_test_reg, preds)
r2 = r2_score(y_test_reg, preds)

print(f"Tuned LightGBM (test): RMSE={rmse:.2f}  MAE={mae:.2f}  R2={r2:.3f}")

with mlflow.start_run(run_name="LightGBM_tuned"):
    mlflow.log_params(search.best_params_)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    signature = infer_signature(X_test_reg_scaled, preds)
    mlflow.lightgbm.log_model(best_lgbm, name="model", signature=signature)

Tuned LightGBM (test): RMSE=33.37  MAE=23.22  R2=0.971
🏃 View run LightGBM_tuned at: http://127.0.0.1:5000/#/experiments/4/runs/9d7b35bb10a14de09c466de523c2df6a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [27]:
mlflow.register_model(
    "runs:/9d7b35bb10a14de09c466de523c2df6a/model",
    "real_estate_regression_model"
)

Successfully registered model 'real_estate_regression_model'.
2026/08/25 19:51:37 WARNING mlflow.tracking._model_registry.fluent: Run with id 9d7b35bb10a14de09c466de523c2df6a has no artifacts at artifact path 'model', registering model based on models:/m-0961365c37104ca99cc637cbd969368b instead
2026/08/25 19:51:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: real_estate_regression_model, version 1
Created version '1' of model 'real_estate_regression_model'.


<ModelVersion: aliases=[], creation_timestamp=1787667697774, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1787667697774, metrics=None, model_id=None, name='real_estate_regression_model', params=None, run_id='9d7b35bb10a14de09c466de523c2df6a', run_link='', source='models:/m-0961365c37104ca99cc637cbd969368b', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

### Classification Model

In [29]:
x_clf_cols = ['Size_in_SqFt', 'Price_in_Lakhs', 'Floor_No', 'Total_Floors',
    'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals',
    'Public_Transport_Accessibility', 'Security',
    'City_Ahmedabad', 'City_Amritsar', 'City_Bangalore', 'City_Bhopal',
    'City_Bhubaneswar', 'City_Bilaspur', 'City_Chennai', 'City_Coimbatore',
    'City_Cuttack', 'City_Dehradun', 'City_Durgapur', 'City_Dwarka',
    'City_Faridabad', 'City_Gaya', 'City_Gurgaon', 'City_Guwahati',
    'City_Haridwar', 'City_Hyderabad', 'City_Indore', 'City_Jaipur',
    'City_Jamshedpur', 'City_Jodhpur', 'City_Kochi', 'City_Kolkata',
    'City_Lucknow', 'City_Ludhiana', 'City_Mangalore', 'City_Mumbai',
    'City_Mysore', 'City_Nagpur', 'City_New_Delhi', 'City_Noida',
    'City_Patna', 'City_Pune', 'City_Raipur', 'City_Ranchi', 'City_Silchar',
    'City_Surat', 'City_Trivandrum', 'City_Vijayawada',
    'City_Vishakhapatnam', 'City_Warangal', 'Property_Type_Apartment',
    'Property_Type_Independent_House', 'Property_Type_Villa', 'Facing_East',
    'Facing_North', 'Facing_South', 'Facing_West',
    'Furnished_Status_Furnished', 'Furnished_Status_Semi_Furnished',
    'Furnished_Status_Unfurnished', 'Owner_Type_Broker',
    'Owner_Type_Builder', 'Owner_Type_Owner']
X_clf = df[x_clf_cols]
y_clf = df['Good_Investment']

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

In [32]:
scaler_clf = StandardScaler()
X_train_clf_scaled = pd.DataFrame(scaler_clf.fit_transform(X_train_clf),
                                  columns=X_train_clf.columns, index=X_train_clf.index)
X_test_clf_scaled = pd.DataFrame(scaler_clf.transform(X_test_clf),
                                 columns=X_test_clf.columns, index=X_test_clf.index)

In [31]:
mlflow.set_experiment("real_estate_classification")

clf_models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
}

2026/08/25 20:00:23 INFO mlflow.tracking.fluent: Experiment with name 'real_estate_classification' does not exist. Creating a new experiment.


In [33]:
for name, model in clf_models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_clf_scaled, y_train_clf)
        preds = model.predict(X_test_clf_scaled)
        proba = model.predict_proba(X_test_clf_scaled)[:, 1]

        acc = accuracy_score(y_test_clf, preds)
        prec = precision_score(y_test_clf, preds)
        rec = recall_score(y_test_clf, preds)
        f1 = f1_score(y_test_clf, preds)
        auc = roc_auc_score(y_test_clf, proba)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("roc_auc", auc)

        signature = infer_signature(X_test_clf_scaled, preds)
        if name == "XGBoost":
            mlflow.xgboost.log_model(model, name="model", signature=signature)
        elif name == "LightGBM":
            mlflow.lightgbm.log_model(model, name="model", signature=signature)
        else:
            mlflow.sklearn.log_model(model, name="model", signature=signature)

        print(f"{name}: Acc={acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f} AUC={auc:.3f}")

LogisticRegression: Acc=0.714 Prec=0.446 Rec=0.162 F1=0.237 AUC=0.723
🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/5/runs/3978be497c0b4163a5a23028b5301ceb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
KNN: Acc=0.684 Prec=0.365 Rec=0.197 F1=0.256 AUC=0.576
🏃 View run KNN at: http://127.0.0.1:5000/#/experiments/5/runs/bddee13131464ba5a7931433fd0a0a2d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
DecisionTree: Acc=0.660 Prec=0.385 Rec=0.393 F1=0.389 AUC=0.577
🏃 View run DecisionTree at: http://127.0.0.1:5000/#/experiments/5/runs/29d1065bce8d4a99a44289f4ae276710
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
RandomForest: Acc=0.717 Prec=0.429 Rec=0.086 F1=0.143 AUC=0.721
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/5/runs/0407dd3962dc48cd8b5ea85e43230319
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
GradientBoosting: Acc=0.723 Prec=0.428 Rec=0.015 F1=0.029 AUC=0.722
🏃 View run GradientBoostin